# 🔀 하이브리드 검색 (Hybrid Search) 실습

키워드 검색과 벡터 검색을 결합하여 최고의 검색 품질을 제공하는 방법을 학습합니다.

## 📋 학습 내용

1. **기본 하이브리드 검색**: 키워드 + 벡터 결합
2. **RRF 이해**: Reciprocal Rank Fusion 동작 원리
3. **비교 분석**: 키워드 vs 벡터 vs 하이브리드
4. **실무 시나리오**: 다양한 검색 패턴 적용
5. **하이브리드 + 필터**: 조건과 함께 사용

## 🔍 하이브리드 검색의 장점

### RRF (Reciprocal Rank Fusion)
- 키워드 검색(BM25)과 벡터 검색의 순위를 결합
- 두 방식에서 모두 높은 순위를 받은 결과가 상위로
- 키워드와 의미 모두 고려한 최적의 검색

### 공식
```
RRF 점수 = Σ(1 / (k + rank))
```
- k: 상수 (기본값 60)
- rank: 각 검색 방식에서의 순위

---

## 1️⃣ 라이브러리 임포트 및 초기화

In [1]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
search_endpoint = os.getenv("SEARCH_ENDPOINT")
search_key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

# Azure OpenAI 설정
openai_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
openai_key = os.getenv("FOUNDRY_PROJECT_KEY")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# SearchClient 초기화
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key)
)

# OpenAI Client 초기화
openai_client = AzureOpenAI(
    azure_endpoint=openai_endpoint,
    api_key=openai_key,
    api_version=api_version
)

print("✅ 클라이언트 초기화 완료")
print(f"   Endpoint: {search_endpoint}")
print(f"   Index: {index_name}")

✅ 클라이언트 초기화 완료
   Endpoint: https://foundryiq-aisearch-260114-changjuahn.search.windows.net
   Index: products-index


## 2️⃣ 유틸리티 함수 정의

In [2]:
def get_embedding(text):
    """
    텍스트를 벡터로 변환
    """
    response = openai_client.embeddings.create(
        input=text,
        model=embedding_deployment
    )
    return response.data[0].embedding

def print_results(results, title="검색 결과", show_score=True, show_details=False):
    """
    검색 결과를 보기 좋게 출력
    """
    print(f"\n{'='*80}")
    print(f"{title}")
    print('='*80)
    
    result_list = list(results)
    if not result_list:
        print("검색 결과가 없습니다.")
        return result_list
    
    for idx, result in enumerate(result_list, 1):
        if show_score and '@search.score' in result:
            score = result['@search.score']
            print(f"\n{idx}. [점수: {score:.4f}] {result.get('title', 'N/A')}")
        else:
            print(f"\n{idx}. {result.get('title', 'N/A')}")
        
        print(f"   브랜드: {result.get('brand', 'N/A')} | 카테고리: {result.get('category', 'N/A')}")
        print(f"   가격: {result.get('normal_price', 0):,}원")
        
        if show_details and 'review' in result and result['review']:
            review = result['review'][:100]
            print(f"   리뷰: {review}...")
    
    return result_list

print("✅ 유틸리티 함수 정의 완료")

✅ 유틸리티 함수 정의 완료


---

## 3️⃣ 키워드 vs 벡터 vs 하이브리드 비교

같은 검색어로 세 가지 방식을 비교하여 하이브리드 검색의 장점을 이해합니다.

In [27]:
query_text = "러닝화"

print(f"\n{'='*80}")
print(f"🔍 비교 검색어: '{query_text}'")
print('='*80)
print("\n세 가지 검색 방식의 결과를 비교합니다:")
print("1️⃣  키워드 검색 (BM25) - 정확한 단어 매칭")
print("2️⃣  벡터 검색 (Semantic) - 의미 기반 검색")
print("3️⃣  하이브리드 검색 (RRF) - 두 방식 결합\n")

# 벡터 생성
query_vector = get_embedding(query_text)

# 1. 키워드 검색만 (BM25)
print("\n" + "="*80)
print("1️⃣  키워드 검색 (BM25)")
print("="*80)
print("💡 '러닝화'와 '운동화' 단어가 정확히 포함된 제품 검색")

keyword_results = search_client.search(
    search_text=query_text,
    select=["title", "brand", "category", "normal_price"],
    top=5
)

keyword_list = print_results(keyword_results, title="📝 키워드 검색 결과")

# 2. 벡터 검색만 (Semantic)
print("\n" + "="*80)
print("2️⃣  벡터 검색 (Semantic)")
print("="*80)
print("💡 의미적으로 유사한 제품 검색 (브랜드명이 정확히 없어도 가능)")

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="content_vector"
)

vector_results = search_client.search(
    search_text=None,  # 키워드 검색 비활성화
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price"],
    top=5
)

vector_list = print_results(vector_results, title="🎯 벡터 검색 결과")

# 3. 하이브리드 검색 (Keyword + Vector)
print("\n" + "="*80)
print("3️⃣  하이브리드 검색 (RRF)")
print("="*80)
print("💡 키워드 매칭과 의미 유사도를 모두 고려한 최적의 결과")

#차이는 Seasrch Text와 Vector query를 모두 사용하는 것
hybrid_results = search_client.search(
    search_text=query_text,  # 키워드 검색 활성화
    vector_queries=[vector_query],  # 벡터 검색 추가
    select=["title", "brand", "category", "normal_price"],
    top=5
)

hybrid_list = print_results(hybrid_results, title="🔀 하이브리드 검색 결과")

# 결과 분석
print("\n" + "="*80)
print("📊 결과 분석")
print("="*80)

keyword_titles = set([r['title'] for r in keyword_list])
vector_titles = set([r['title'] for r in vector_list])
hybrid_titles = set([r['title'] for r in hybrid_list])

print(f"\n📝 키워드 검색: {len(keyword_titles)}개 제품")
print(f"   - '러닝화' 브랜드가 정확히 포함된 제품")
print(f"   - 장점: 정확한 브랜드명 매칭")
print(f"   - 단점: 유사 브랜드나 대체 제품 찾기 어려움")

print(f"\n🎯 벡터 검색: {len(vector_titles)}개 제품")
print(f"   - 의미적으로 유사한 운동화 제품")
print(f"   - 장점: 자연스러운 검색, 유사 제품 발견")
print(f"   - 단점: 정확한 브랜드 지정 약함")

print(f"\n🔀 하이브리드 검색: {len(hybrid_titles)}개 제품")
print(f"   - 키워드와 벡터 점수를 RRF로 통합")
print(f"   - 장점: 정확한 브랜드 + 의미 유사도 모두 고려")
print(f"   - 결과: 두 방식의 장점을 결합한 최적의 검색")

# 결과 중복 분석
only_keyword = keyword_titles - vector_titles - hybrid_titles
only_vector = vector_titles - keyword_titles - hybrid_titles
in_hybrid = keyword_titles & vector_titles & hybrid_titles

print(f"\n🔍 결과 중복 분석:")
print(f"   - 키워드만: {len(only_keyword)}개")
print(f"   - 벡터만: {len(only_vector)}개")
print(f"   - 세 방식 모두: {len(in_hybrid)}개")
print(f"   → 하이브리드가 가장 포괄적이고 정확한 결과 제공")


🔍 비교 검색어: '러닝화'

세 가지 검색 방식의 결과를 비교합니다:
1️⃣  키워드 검색 (BM25) - 정확한 단어 매칭
2️⃣  벡터 검색 (Semantic) - 의미 기반 검색
3️⃣  하이브리드 검색 (RRF) - 두 방식 결합


1️⃣  키워드 검색 (BM25)
💡 '러닝화'와 '운동화' 단어가 정확히 포함된 제품 검색

📝 키워드 검색 결과

1. [점수: 5.2116] [그랜드스테이지] 써코니 허리케인 24 M S20933-500
   브랜드: 그랜드스테이지 | 카테고리: 스포츠/레져
   가격: 118,680원

2. [점수: 4.5102] [그랜드스테이지] 써코니 엔돌핀 프로 4 M S20939-30
   브랜드: 그랜드스테이지 | 카테고리: 패션잡화
   가격: 247,480원

3. [점수: 3.7567] [아디다스코리아공식] JQ0593 도쿄 W 
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 139,000원

4. [점수: 2.2020] [아디다스코리아공식] JS4957 아디제로 보스턴 13 W
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 179,000원

5. [점수: 1.8553] [그랜드스테이지] 써코니 엔돌핀 프로 4 M S20939-201
   브랜드: 그랜드스테이지 | 카테고리: 패션잡화
   가격: 247,480원

2️⃣  벡터 검색 (Semantic)
💡 의미적으로 유사한 제품 검색 (브랜드명이 정확히 없어도 가능)

🎯 벡터 검색 결과

1. [점수: 0.5625] 설화수[8월]에센셜 데일리 세트 (자음2종)
   브랜드: 설화수 | 카테고리: 이미용
   가격: 150,000원

2. [점수: 0.5601] 리케이 전동 발톱 다듬기 RK-5300  
   브랜드: 리케이 | 카테고리: 문화/취미
   가격: 53,500원

3. [점수: 0.5599] 설화수[8월]퍼펙팅 쿠션 에어리 본품15g+리필15g SPF50+
   브랜드: 설화수 | 카테고리: 이미용
   가격: 89,

---

## 🧮 RRF (Reciprocal Rank Fusion) 동작 원리

방금 실행한 하이브리드 검색에서 **RRF가 어떻게 점수를 계산**하는지 자세히 알아봅니다.

### RRF란?

RRF는 **여러 검색 방식의 순위를 통합**하여 최종 점수를 계산하는 알고리즘입니다.

### 계산 공식

```
RRF 점수 = Σ(1 / (k + rank))
```

| 파라미터 | 설명 | 기본값 |
|----------|------|--------|
| **k** | 스무딩 상수 (낮은 순위의 영향력 조절) | 60 (고정) |
| **rank** | 각 검색 방식에서의 순위 (1부터 시작) | - |

> ⚠️ **참고**: Azure AI Search에서 k=60은 **변경할 수 없는 고정 상수**입니다.  
> 실험적으로 최적의 성능을 보이는 값으로 확인되어 내부적으로 고정되어 있습니다.

---

### 📊 실제 데이터 예시: "러닝화" 검색

#### 1️⃣ 키워드 검색 (BM25) 결과

"러닝화"라는 **단어가 포함**된 제품 순서대로 정렬:

| 순위 | 제품명 | 이유 |
|------|--------|------|
| 1위 | 나이키 에어줌 러닝화 | "러닝화" 단어 정확히 포함 |
| 2위 | 아디다스 울트라부스트 러닝화 | "러닝화" 단어 포함 |
| 3위 | 뉴발란스 퓨어셀 러닝화 | "러닝화" 단어 포함 |
| ❌ | 아식스 젤카야노 30 | "러닝화" 단어 없음 (미검색) |

#### 2️⃣ 벡터 검색 (Semantic) 결과

"러닝화"와 **의미적으로 유사**한 제품 순서대로 정렬:

| 순위 | 제품명 | 이유 |
|------|--------|------|
| 1위 | 아식스 젤카야노 30 | 의미상 러닝/조깅 관련 최고 유사도 |
| 2위 | 나이키 에어줌 러닝화 | 러닝 관련 제품 |
| 3위 | 호카 클리프턴 9 | 러닝화 카테고리 |
| 4위 | 아디다스 울트라부스트 러닝화 | 러닝 관련 제품 |

#### 3️⃣ RRF 통합 결과 (k=60)

| 제품명 | 키워드 순위 | 벡터 순위 | RRF 점수 계산 | 최종 순위 |
|--------|------------|----------|---------------|-----------|
| 나이키 에어줌 러닝화 | 1위 | 2위 | 1/(60+1) + 1/(60+2) = 0.0164 + 0.0161 = **0.0325** | 🥇 1위 |
| 아식스 젤카야노 30 | ❌ 없음 | 1위 | 0 + 1/(60+1) = **0.0164** | 🥈 2위 |
| 아디다스 울트라부스트 러닝화 | 2위 | 4위 | 1/(60+2) + 1/(60+4) = 0.0161 + 0.0156 = **0.0317** | 🥉 3위 |
| 뉴발란스 퓨어셀 러닝화 | 3위 | ❌ 없음 | 1/(60+3) + 0 = **0.0159** | 4위 |
| 호카 클리프턴 9 | ❌ 없음 | 3위 | 0 + 1/(60+3) = **0.0159** | 5위 |

---

### 💡 결과 해석

| 제품 | 분석 |
|------|------|
| **나이키 에어줌 러닝화** | 키워드 1위 + 벡터 2위 → **양쪽 모두 강함** → 최종 1위 ⭐ |
| **아식스 젤카야노 30** | 키워드 ❌ + 벡터 1위 → **의미는 최고지만 단어 없음** → 2위 |
| **아디다스 울트라부스트** | 키워드 2위 + 벡터 4위 → **양쪽 모두 있음** → 3위 |

### 🎯 RRF의 핵심 포인트

1. ✅ **"나이키 에어줌 러닝화"**: 두 방식 모두 상위 → **최종 1위**
2. ✅ **"아식스 젤카야노 30"**: 벡터만 1위지만 키워드에 없어도 → **결과에 포함**
3. ✅ **순위 기반**: BM25 점수와 벡터 유사도의 **스케일 차이 문제 해결**
4. ✅ **k=60**: 순위 간 점수 차이를 완만하게 → **안정적인 결과**

---

### ⚖️ 벡터 가중치 (Vector Weighting)로 비중 조절

k 값은 변경할 수 없지만, **벡터 쿼리의 `weight` 파라미터**로 키워드와 벡터의 비중을 조절할 수 있습니다.

#### 사용 방법

```python
# 벡터 비중 높이기 (의미 검색 우선)
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector",
    weight=2.0  # 벡터 비중 2배 증가
)

# 키워드 비중 높이기 (정확한 단어 매칭 우선)
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector",
    weight=0.5  # 벡터 비중 절반 → 상대적으로 키워드 비중 증가
)
```

#### weight 값에 따른 비중 변화

| weight 값 | 키워드 vs 벡터 | 사용 시나리오 |
|-----------|---------------|--------------|
| **0.3** | 키워드 ⬆️⬆️⬆️ / 벡터 ⬇️ | 제품 코드, 모델명 검색 |
| **0.5** | 키워드 ⬆️⬆️ / 벡터 ⬇️ | 브랜드명 + 간단한 설명 |
| **1.0** (기본값) | 키워드 = 벡터 | 균형 잡힌 하이브리드 |
| **1.5** | 키워드 ⬇️ / 벡터 ⬆️ | 자연어 질문 검색 |
| **2.0** | 키워드 ⬇️ / 벡터 ⬆️⬆️ | "~같은 제품 찾아줘" |

#### 💡 실무 가이드

| 검색 유형 | 권장 weight | 이유 |
|----------|-------------|------|
| 정확한 브랜드/모델명 | 0.3 ~ 0.5 | 키워드 매칭이 중요 |
| 기술 용어 + 설명 | 0.7 ~ 1.0 | 균형 필요 |
| 자연어 질문 | 1.5 ~ 2.0 | 의미 이해가 중요 |
| 유사 제품 추천 | 2.0 이상 | 벡터 유사도 우선 |

---

### ⚙️ Azure AI Search의 RRF

```python
# Azure AI Search에서 하이브리드 검색 시 자동으로 RRF 적용
results = search_client.search(
    search_text="러닝화",        # 키워드 검색 (BM25)
    vector_queries=[...],        # 벡터 검색
    # → 두 결과가 RRF로 자동 통합!
)
```

---

---

## 4️⃣ 실무 시나리오: 브랜드 + 의미 검색

사용자가 특정 브랜드의 제품을 자연어로 검색하는 시나리오입니다.

In [29]:
test_queries = [
    ("노스페이스 바람막이", "정확한 브랜드 + 제품 카테고리"),
    ("설화수 에센스", "브랜드 + 제품 특성"),
    ("아디다스 러닝화", "브랜드 + 사용 목적"),
    ("롱샴 숄더백", "브랜드 + 제품 특징")
]

for query, description in test_queries:
    print(f"\n{'='*80}")
    print(f"🔍 검색어: '{query}'")
    print(f"📝 시나리오: {description}")
    print('='*80)
    
    query_vector = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector"
    )
    
    # 하이브리드 검색
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["title", "brand", "category", "normal_price"],
        top=3
    )
    
    for idx, result in enumerate(results, 1):
        score = result.get('@search.score', 0)
        print(f"\n{idx}. [점수: {score:.4f}] {result['title']}")
        print(f"   브랜드: {result['brand']} | 카테고리: {result['category']}")
        print(f"   가격: {result['normal_price']:,}원")


🔍 검색어: '노스페이스 바람막이'
📝 시나리오: 정확한 브랜드 + 제품 카테고리

1. [점수: 0.0333] 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 65,550원

2. [점수: 0.0325] 노스페이스 NJ2HR11 남성 패커블 LT 자켓 (남성 방수 바람막이 자켓)
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 155,800원

3. [점수: 0.0325] 노스페이스 NJ2HR41 여성 패커블 LT 자켓  (여성 바람막이 방수자켓)
   브랜드: 노스페이스 | 카테고리: 스포츠/레져
   가격: 155,800원

🔍 검색어: '설화수 에센스'
📝 시나리오: 브랜드 + 제품 특성

1. [점수: 0.0331] 설화수[8월]에센셜 데일리 세트 (자음2종)
   브랜드: 설화수 | 카테고리: 이미용
   가격: 150,000원

2. [점수: 0.0331] 설화수[8월]윤조에센스 6세대 90ml 기획세트
   브랜드: 설화수 | 카테고리: 이미용
   가격: 140,000원

3. [점수: 0.0323] 설화수[8월]퍼펙팅 쿠션 에어리 본품15g+리필15g SPF50+
   브랜드: 설화수 | 카테고리: 이미용
   가격: 89,000원

🔍 검색어: '아디다스 러닝화'
📝 시나리오: 브랜드 + 사용 목적

1. [점수: 0.0328] [아디다스코리아공식] JQ0593 도쿄 W 
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 139,000원

2. [점수: 0.0328] [아디다스코리아공식] JS4957 아디제로 보스턴 13 W
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 179,000원

3. [점수: 0.0325] [아디다스코리아공식] JS4939 아디제로 보스턴 13 
   브랜드: 아디다스 | 카테고리: 스포츠/레져
   가격: 179,000원

🔍 검색어: '롱샴 숄더백'
📝 시나리오: 브랜드 

---

## 5️⃣ 실무 시나리오: 제품 특성 + 자연어

제품 특성과 사용자의 자연어 표현이 섞인 검색 시나리오입니다.

In [30]:
query_text = "신생아 출산 선물 추천"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 특징:")
print("   - '신생아', '출산': 정확한 카테고리 용어 (키워드 매칭 중요)")
print("   - '선물 추천': 자연어 표현 (의미 이해 필요)")
print("   - 하이브리드가 두 요구사항을 모두 충족\n")

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price"],
    top=5
)

print_results(results, title="🔀 하이브리드 검색 결과")

print("\n💡 하이브리드의 강점:")
print("   ✅ '신생아', '출산' 키워드가 포함된 제품 우선")
print("   ✅ '선물 추천'이라는 의미를 이해하여 선물용 제품 검색")
print("   ✅ 정확한 용어와 사용자 니즈를 모두 고려")


🔍 검색어: '신생아 출산 선물 추천'

💡 특징:
   - '신생아', '출산': 정확한 카테고리 용어 (키워드 매칭 중요)
   - '선물 추천': 자연어 표현 (의미 이해 필요)
   - 하이브리드가 두 요구사항을 모두 충족


🔀 하이브리드 검색 결과

1. [점수: 0.0323] 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 39,000원

2. [점수: 0.0315] [M밍크뮤D] 26년말띠(WH)드림코니리오셀 배내저고리양말손싸개세트 (35A700110301) 출산준비물/신생아선물/
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 77,600원

3. [점수: 0.0314] 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건/10장/신생아/출산용품/선물
   브랜드: 블루독베이비 | 카테고리: 유아동
   가격: 22,190원

4. [점수: 0.0308] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원

5. [점수: 0.0305] [압소바1] 곰고무연사목욕타올+손수건+인형세트 MIAZA-10911 [임신/출산선물]
   브랜드: 압소바 | 카테고리: 유아동
   가격: 96,000원

💡 하이브리드의 강점:
   ✅ '신생아', '출산' 키워드가 포함된 제품 우선
   ✅ '선물 추천'이라는 의미를 이해하여 선물용 제품 검색
   ✅ 정확한 용어와 사용자 니즈를 모두 고려


---

## 6️⃣ 실무 시나리오: 카테고리 + 품질 표현

구체적인 카테고리와 품질을 나타내는 표현이 결합된 검색입니다.

In [31]:
query_text = "고급스러운 실크 스카프"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 특징:")
print("   - '실크 스카프': 구체적인 제품 카테고리 (키워드)")
print("   - '고급스러운': 품질을 나타내는 표현 (의미)")
print("   - 하이브리드로 두 요소를 모두 반영\n")

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

# 키워드 검색만
print("\n" + "="*80)
print("📝 키워드 검색만")
print("="*80)
keyword_results = search_client.search(
    search_text=query_text,
    select=["title", "brand", "category", "normal_price"],
    top=5
)
print_results(keyword_results, title="'실크 스카프' 키워드 포함 제품")

# 하이브리드 검색
print("\n" + "="*80)
print("🔀 하이브리드 검색")
print("="*80)
hybrid_results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price"],
    top=5
)
print_results(hybrid_results, title="'실크 스카프' + '고급스러운' 의미 반영")

print("\n💡 차이점:")
print("   - 키워드만: '실크 스카프' 단어만 확인")
print("   - 하이브리드: '고급스러운'의 의미도 반영 (프리미엄 브랜드, 높은 가격대)")


🔍 검색어: '고급스러운 실크 스카프'

💡 특징:
   - '실크 스카프': 구체적인 제품 카테고리 (키워드)
   - '고급스러운': 품질을 나타내는 표현 (의미)
   - 하이브리드로 두 요소를 모두 반영


📝 키워드 검색만

'실크 스카프' 키워드 포함 제품

1. [점수: 21.1641] [막스마라] 까레 실크 스카프 / 라이트 블루
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

2. [점수: 21.0518] [막스마라] 까레 실크 스카프 / 그린
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

3. [점수: 19.5856] [헤지스핸드백] HISC5E342 베이지 프린트 실크 스카프
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 60,480원

4. [점수: 18.9126] [헤지스핸드백] HISC5E311 Toile de Jouy 네이비 쁘띠 실크 스카프
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 43,680원

5. [점수: 11.5448] [파리게이츠] (521D2AC501)여성 폼폼 로고 패턴 스퀘어 스카프
   브랜드: 파리게이츠 | 카테고리: 스포츠/레져
   가격: 27,360원

🔀 하이브리드 검색

'실크 스카프' + '고급스러운' 의미 반영

1. [점수: 0.0328] [헤지스핸드백] HISC5E342 베이지 프린트 실크 스카프
   브랜드: 헤지스핸드백 | 카테고리: 패션잡화
   가격: 60,480원

2. [점수: 0.0328] [막스마라] 까레 실크 스카프 / 라이트 블루
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

3. [점수: 0.0328] [막스마라] 까레 실크 스카프 / 그린
   브랜드: Max Mara | 카테고리: 패션잡화
   가격: 680,000원

4. [점수: 0.0315] [헤지스핸드백] HISC5E311 Toile de Jouy 네이비 쁘띠 실크 스카

---

## 7️⃣ 실무 시나리오: 제품 + 사용 목적

제품 유형과 사용 목적이 결합된 검색입니다.

In [32]:
query_text = "야외 활동용 경량 백팩"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 특징:")
print("   - '야외 활동용': 사용 목적 (의미)")
print("   - '경량 백팩': 제품 특성 + 카테고리 (키워드)")
print("   - 하이브리드로 등산/캠핑용 가방 검색\n")

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price"],
    top=5
)

print_results(results, title="🔀 하이브리드 검색 결과")

print("\n💡 하이브리드의 강점:")
print("   ✅ '백팩' 카테고리 제품 검색")
print("   ✅ '야외 활동용', '경량'의 의미를 이해 (등산, 캠핑, 가벼운 소재 등)")
print("   ✅ 사용 목적에 맞는 제품 추천")


🔍 검색어: '야외 활동용 경량 백팩'

💡 특징:
   - '야외 활동용': 사용 목적 (의미)
   - '경량 백팩': 제품 특성 + 카테고리 (키워드)
   - 하이브리드로 등산/캠핑용 가방 검색


🔀 하이브리드 검색 결과

1. [점수: 0.0331] 디스커버리 25N 신상품 등산배낭 DXBK5025N 에어 스플리트 백팩 17L 경량가방
   브랜드: 디스커버리 | 카테고리: 스포츠/레져
   가격: 129,000원

2. [점수: 0.0315] [윌슨] 드로우스트링 백팩 V1 W253006TBP72 NVY
   브랜드: 윌슨(백화점) | 카테고리: 스포츠/레져
   가격: 151,050원

3. [점수: 0.0306] [윌슨] 드로우스트링 백팩 V2 W251006TBP13 WHT
   브랜드: 윌슨(백화점) | 카테고리: 스포츠/레져
   가격: 151,050원

4. [점수: 0.0295] BAMKEL 밤켈 모던 42QT 하드쿨러
   브랜드: 밤켈 | 카테고리: 스포츠/레져
   가격: 230,000원

5. [점수: 0.0283] BAMKEL 밤켈 모던 22QT 하드쿨러
   브랜드: 밤켈 | 카테고리: 스포츠/레져
   가격: 180,000원

💡 하이브리드의 강점:
   ✅ '백팩' 카테고리 제품 검색
   ✅ '야외 활동용', '경량'의 의미를 이해 (등산, 캠핑, 가벼운 소재 등)
   ✅ 사용 목적에 맞는 제품 추천


---

## 8️⃣ 하이브리드 + 필터 검색

하이브리드 검색에 필터 조건을 추가하여 더욱 정확한 검색이 가능합니다.

In [33]:
query_text = "출산 선물 추천"
category_filter = "유아동"
max_price = 100000

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print(f"🎯 필터 조건:")
print(f"   - 카테고리: {category_filter}")
print(f"   - 최대 가격: {max_price:,}원 이하")
print('='*80)

query_vector = get_embedding(query_text)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=50,  # 필터 전 충분한 결과 확보
    fields="content_vector"
)

filter_expression = f"category eq '{category_filter}' and normal_price le {max_price}"

results = search_client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    filter=filter_expression,
    select=["title", "brand", "category", "normal_price"],
    top=10
)

result_list = print_results(results, title="🔀 하이브리드 + 필터 검색 결과")

print("\n💡 하이브리드 + 필터의 강점:")
print("   ✅ 키워드 매칭: '출산', '선물' 정확히 포함")
print("   ✅ 의미 이해: 유아동 관련 제품 검색")
print("   ✅ 카테고리 필터: 정확히 유아동 제품만")
print("   ✅ 가격 필터: 예산 범위 내")
print(f"   → 네 가지 조건을 모두 만족하는 최적의 결과: {len(result_list)}개")


🔍 검색어: '출산 선물 추천'
🎯 필터 조건:
   - 카테고리: 유아동
   - 최대 가격: 100,000원 이하

🔀 하이브리드 + 필터 검색 결과

1. [점수: 0.0325] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원

2. [점수: 0.0325] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동
   가격: 40,000원

3. [점수: 0.0320] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 38,000원

4. [점수: 0.0310] 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(35W70-BHB-01/35A70-011-03)
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 75,200원

5. [점수: 0.0308] [M밍크뮤D] 26년말띠(WH)드림코니리오셀 배내저고리양말손싸개세트 (35A700110301) 출산준비물/신생아선물/
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 77,600원

6. [점수: 0.0304] (에뜨와) (말띠해출산선물)헨리우주복V블로니양말(2종세트) (07S71750H) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동
   가격: 98,000원

7. [점수: 0.0301] [노베즈][선물포장] 촉촉3종세트 (로션+워시+크림) 선물추천 스킨케어선물 화장품세트
   브랜드: 노베즈 | 카테고리: 유아동
   가격: 81,000원

8. [점수: 0.0294] [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형
   브랜드: 블루독베이비 | 카테고리: 유아동
   가격: 36,000원

9. [점수: 0.0293] 압소

---

## 9️⃣ 하이브리드 + 멀티 벡터 검색

하이브리드 검색에 여러 벡터 필드를 동시에 활용할 수 있습니다.

In [34]:
query_text = "배송 빠르고 품질 좋은 제품"

print(f"\n{'='*80}")
print(f"🔍 검색어: '{query_text}'")
print('='*80)
print("\n💡 전략:")
print("   - 키워드: '배송', '품질', '제품' 단어 매칭")
print("   - content_vector: 제품 정보 검색")
print("   - review_vector: 리뷰에서 배송/품질 언급 검색")
print("   - 세 가지 신호를 모두 활용\n")

query_vector = get_embedding(query_text)

# 벡터 쿼리 1: content_vector
content_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

# 벡터 쿼리 2: review_vector
review_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="review_vector"
)

results = search_client.search(
    search_text=query_text,
    vector_queries=[content_query, review_query],
    select=["title", "brand", "category", "normal_price", "review"],
    top=5
)

print_results(results, title="🔀 하이브리드 + 멀티벡터 검색 결과", show_details=True)

print("\n💡 최강의 조합:")
print("   ✅ 키워드 매칭: 검색어 단어 포함")
print("   ✅ 제품 정보 벡터: 제품 자체의 관련성")
print("   ✅ 리뷰 벡터: 실제 사용자 평가 반영")
print("   → 가장 포괄적이고 정확한 검색 결과")


🔍 검색어: '배송 빠르고 품질 좋은 제품'

💡 전략:
   - 키워드: '배송', '품질', '제품' 단어 매칭
   - content_vector: 제품 정보 검색
   - review_vector: 리뷰에서 배송/품질 언급 검색
   - 세 가지 신호를 모두 활용


🔀 하이브리드 + 멀티벡터 검색 결과

1. [점수: 0.0437] [M밍크뮤D] (WH)루루뱀부손수건SET 10장세트 (3417000603) 출산준비물/신생아선물/배내수트/배내저고리
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 20,880원
   리뷰: 밍크뮤 루루뱀부손수건 10장 세트는 신생아 피부에 정말 순하고 부드러워서 만족스러웠습니다. 뱀부 소재라서 흡수력도 좋고 빨아도 쉽게 헤지지 않아 실용적이에요. 크기도 적당해서 목에...

2. [점수: 0.0425] [노베즈][선물포장] 촉촉3종세트 (로션+워시+크림) 선물추천 스킨케어선물 화장품세트
   브랜드: 노베즈 | 카테고리: 유아동
   가격: 81,000원
   리뷰: 아이 피부가 예민해서 순한 제품을 찾다가 노베즈 촉촉3종세트를 구매했어요. 로션과 크림 모두 끈적임 없이 산뜻하게 발리면서도 보습력이 뛰어나서 아침저녁으로 발라주니 피부가 훨씬 촉...

3. [점수: 0.0377] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   브랜드: 밍크뮤 | 카테고리: 유아동
   가격: 35,280원
   리뷰: 조카 출산 선물로 밍크뮤 드림루루 딸랑이세트를 구매했어요. 부드러운 소재와 안전한 디자인 덕분에 아기가 안심하고 가지고 놀더라고요. 크기도 적당해서 아기 손에 딱 맞고, 소리도 자...

4. [점수: 0.0310] 모던비스트 레드 홀리데이 장난감 세트 키티 MDB015RD
   브랜드: 모던비스트 | 카테고리: 문화/취미
   가격: 70,000원
   리뷰: 모던비스트 레드 홀리데이 장난감 세트 키티 MDB015RD를 구매했습니다. 모던비스트 제품답게 품질이 

---

## 🔟 실무 패턴: 다양한 검색 시나리오

실무에서 자주 발생하는 다양한 검색 패턴을 테스트합니다.

In [35]:
scenarios = [
    ("친구 생일 선물 추천", "자연어 질문 + 추천 요청"),
    ("통기성 좋은 여름용 모자", "기능 + 계절 + 제품"),
    ("강아지 건강 영양제", "반려동물 + 제품 특성"),
    ("라코스테 폴로셔츠", "브랜드 + 제품 카테고리"),
    ("등산용 경량 자켓", "활동 + 제품 특성"),
    ("아기 백일 선물 세트", "대상 + 목적 + 제품")
]

print("\n" + "="*80)
print("🎯 실무 검색 패턴 테스트")
print("="*80)
print("\n하이브리드 검색이 다양한 검색 패턴을 어떻게 처리하는지 확인합니다.\n")

for query, pattern in scenarios:
    print(f"\n{'='*80}")
    print(f"🔍 검색어: '{query}'")
    print(f"📝 패턴: {pattern}")
    print('='*80)
    
    query_vector = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector"
    )
    
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["title", "brand", "category", "normal_price"],
        top=3
    )
    
    result_count = 0
    for idx, result in enumerate(results, 1):
        result_count += 1
        score = result.get('@search.score', 0)
        print(f"\n{idx}. [점수: {score:.4f}] {result['title']}")
        print(f"   {result['brand']} | {result['category']} | {result['normal_price']:,}원")
    
    if result_count == 0:
        print("   검색 결과가 없습니다. (데이터에 해당 제품 없음)")


🎯 실무 검색 패턴 테스트

하이브리드 검색이 다양한 검색 패턴을 어떻게 처리하는지 확인합니다.


🔍 검색어: '친구 생일 선물 추천'
📝 패턴: 자연어 질문 + 추천 요청

1. [점수: 0.0333] [멜리멜로] 생일/집들이선물 클래식 스톤 디퓨저 생일 집들이 친구선물
   멜리멜로 | 인테리어 | 28,000원

2. [점수: 0.0320] [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형
   블루독베이비 | 유아동 | 36,000원

3. [점수: 0.0318] 선물추천 [밍크뮤HP]26년말띠선물 (CR)코니밍크애착인형 양말SET(35W70-BHB-01/35A70-011-03)
   밍크뮤 | 유아동 | 75,200원

🔍 검색어: '통기성 좋은 여름용 모자'
📝 패턴: 기능 + 계절 + 제품

1. [점수: 0.0333] 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)
   노스페이스 | 스포츠/레져 | 65,550원

2. [점수: 0.0325] [오르카홈] 여름용 냉감 중형(M) 에어코일 커브 펫 매트리스 2color 
   오르카홈 | 문화/취미 | 149,000원

3. [점수: 0.0306] 노스페이스 NE3HR55 여성 모자 우먼즈 와이드 햇
   노스페이스 | 스포츠/레져 | 65,550원

🔍 검색어: '강아지 건강 영양제'
📝 패턴: 반려동물 + 제품 특성

1. [점수: 0.0318] 로이코 혼자있는 강아지 고양이 분리불안 예방 자동 노즈워크 장난감 트릿토이 베이직세트
   로이코 | 문화/취미 | 89,000원

2. [점수: 0.0318] 애니먼 강아지 고양이 습식사료 애니캔 플러스 신장·요로계 30g 6개
   애니먼 | 문화/취미 | 12,900원

3. [점수: 0.0318] 피너츠 스누피 하우스 강아지 노즈워크 장난감 세트 (노즈워크+바스락+삑삑이+방울)
   패리스독피너츠 | 문화/취미 | 24,900원

🔍 검색어: '라코스테 폴로셔츠'
📝 패턴: 

---

## ✅ 학습 완료!

축하합니다! 하이브리드 검색을 마스터했습니다.

### 📚 학습한 내용

1. ✅ **하이브리드 검색 원리**: 키워드 + 벡터 결합
2. ✅ **RRF 이해**: Reciprocal Rank Fusion 동작 방식
3. ✅ **세 방식 비교**: 키워드 vs 벡터 vs 하이브리드
4. ✅ **실무 시나리오**: 브랜드, 용어, 의미 결합 검색
5. ✅ **고급 기능**: 필터, 멀티벡터와 결합

### 🎯 하이브리드 검색의 핵심 장점

| 상황 | 키워드 | 벡터 | 하이브리드 |
|------|--------|------|------------|
| 정확한 브랜드명 | ⭐⭐⭐ | ⭐ | ⭐⭐⭐ |
| 자연어 질문 | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| 기술 용어 | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| 의미 이해 | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **종합 평가** | ⭐⭐ | ⭐⭐ | **⭐⭐⭐** |

### 💡 실무 권장사항

**하이브리드 검색 사용 권장:**
- ✅ 전자상거래 제품 검색
- ✅ 문서/기사 검색
- ✅ FAQ/지식베이스
- ✅ 다양한 사용자 검색 패턴

**키워드/벡터만 사용:**
- ⚠️ ID/코드 검색 → 키워드만
- ⚠️ 유사 문서 찾기 → 벡터만
- ⚠️ 성능 최적화 필요 시 → 단일 방식

### 🚀 다음 단계

1. **05-scoring/** - 스코어링 프로파일로 비즈니스 로직 추가
2. **06-re_ranking/** - AI 재순위화로 정확도 극대화
3. **실무 프로젝트** - 실제 서비스에 하이브리드 검색 적용

### 📖 참고 자료

- [Azure AI Search Hybrid Search](https://learn.microsoft.com/azure/search/hybrid-search-overview)
- [Reciprocal Rank Fusion Paper](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [Hybrid Search Best Practices](https://learn.microsoft.com/azure/search/hybrid-search-how-to-query)